In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
cd "../data/Jul"

In [ ]:


# ---- paths ----
red_path   = "S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_Rrs_665.tif"
green_path = "S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_Rrs_559.tif"
blue_path  = "S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_Rrs_492.tif"
out_png    = "L2W_RGB.png"
flags_path = "S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_l2_flags.tif"


In [ ]:
FILL = 9.96921e36  # ACOLITE fill value

# -----------------------------
# Read bands
# -----------------------------
with rasterio.open(red_path) as src:
    red = src.read(1)

with rasterio.open(green_path) as src:
    green = src.read(1)

with rasterio.open(blue_path) as src:
    blue = src.read(1)

with rasterio.open(flags_path) as src:
    flags = src.read(1).astype(np.int32)

# -----------------------------
# Mask fill values
# -----------------------------
red[red >= FILL/10]     = np.nan
green[green >= FILL/10] = np.nan
blue[blue >= FILL/10]   = np.nan

# -----------------------------
# Quality flag masks
# -----------------------------
def has_flag(arr, bit):
    return (arr & (1 << bit)) != 0

mask_out = has_flag(flags, 4)
mask_cirrus     = has_flag(flags, 1)
mask_swir       = has_flag(flags, 0)
mask_negative   = has_flag(flags, 3)
mask_toa   = has_flag(flags, 2)

valid = ~(mask_out |  mask_toa | mask_swir | mask_negative | mask_cirrus)

# -----------------------------
# Stack RGB and mask invalid
# -----------------------------
rgb = np.stack([red, green, blue], axis=-1)
rgb[~valid] = np.nan

# -----------------------------
# Fixed reflectance limits (tuned for coastal waters)
# -----------------------------
rgb[..., 2] = np.clip((blue  - 0.0) / 0.035, 0, 1)  # 490 nm
rgb[..., 1] = np.clip((green - 0.0) / 0.045, 0, 1) # 560 nm
rgb[..., 0] = np.clip((red   - 0.0) / 0.045, 0, 1) # 665 nm


# -----------------------------
# Plot & save 
# -----------------------------
plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title("ACOLITE L2W RGB (water-only stretch)")
plt.axis("off")
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("Rrs_490:", np.nanmin(blue), np.nanmax(blue))
print("Rrs_560:", np.nanmin(green), np.nanmax(green))
print("Rrs_665:", np.nanmin(red), np.nanmax(red))
